In [ ]:
import os
import random
import time
import math
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 시드 고정
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 경로 자동 설정 logic
if os.path.exists('/content/drive/MyDrive'):
    # Colab 환경
    BASE_PATH = '/content/drive/MyDrive/super_solutioner/LNO_base'
else:
    # 로컬 또는 기타 환경
    BASE_PATH = './LNO_base'

DATA_PATH = os.path.join(BASE_PATH, 'data/DIV2K')
CKPT_PATH = os.path.join(BASE_PATH, 'checkpoints_codalno_upgrade')
RESULT_PATH = os.path.join(BASE_PATH, 'results_codalno')

os.makedirs(CKPT_PATH, exist_ok=True)
os.makedirs(RESULT_PATH, exist_ok=True)

print(f"Device: {device} | Base Path: {BASE_PATH}")

In [ ]:
import random
import torch.nn.functional as F

class DIV2KDataset(Dataset):
    # scale과 patch_size 대신, 최종 HR 크기(hr_crop_size)만 받도록 수정 (예: 48 * 4 = 192)
    def __init__(self, root_dir, phase='train', hr_crop_size=192):
        self.phase = phase
        self.hr_crop_size = hr_crop_size

        subset = 'DIV2K_train_HR' if phase == 'train' else 'DIV2K_valid_HR'
        self.image_paths = sorted([os.path.join(root_dir, subset, f) for f in os.listdir(os.path.join(root_dir, subset)) if f.endswith('.png')])
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        hr_img = Image.open(self.image_paths[idx]).convert('RGB')

        # Train: Random Crop
        if self.phase == 'train':
            w, h = hr_img.size
            hp = self.hr_crop_size # 항상 192x192 크기로 자름

            if w < hp or h < hp:
                hr_img = hr_img.resize((max(w, hp), max(h, hp)), Image.BICUBIC)
                w, h = hr_img.size

            x1 = random.randint(0, w - hp)
            y1 = random.randint(0, h - hp)
            hr_patch = hr_img.crop((x1, y1, x1 + hp, y1 + hp))

            # Augmentation (데이터 8배 뻥튀기 효과)
            if random.random() < 0.5: hr_patch = hr_patch.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() < 0.5: hr_patch = hr_patch.transpose(Image.FLIP_TOP_BOTTOM)
            if random.random() < 0.5: hr_patch = hr_patch.rotate(90) # [추가] 90도 회전

        else:
            # Valid: Center Crop (for memory safety)
            w, h = hr_img.size
            crop = 512
            x1 = (w - crop) // 2
            y1 = (h - crop) // 2
            hr_patch = hr_img.crop((x1, y1, x1 + crop, y1 + crop))

        # [핵심] 여기서는 LR을 만들지 않고 HR 텐서만 반환합니다.
        return self.to_tensor(hr_patch)

In [ ]:
class LaplaceSpatialMixer(nn.Module):
    def __init__(self, dim, modes1=16, modes2=16):
        super().__init__()
        self.modes1, self.modes2 = modes1, modes2
        scale = (1 / (dim * dim))
        self.weights_pole1 = nn.Parameter(scale * torch.rand(1, 1, modes1, dtype=torch.cfloat))
        self.weights_pole2 = nn.Parameter(scale * torch.rand(1, 1, modes2, dtype=torch.cfloat))
        self.weights_residue = nn.Parameter(scale * torch.rand(dim, dim, modes1, modes2, dtype=torch.cfloat))

    def forward(self, x, target_size=None):
        B, C, H, W = x.shape
        if target_size is None: target_size = (H, W)

        alpha = torch.fft.fft2(x, dim=[-2, -1])
        alpha_modes = alpha[..., :self.modes1, :self.modes2]

        omega1 = torch.fft.fftfreq(self.modes1, d=1/self.modes1).to(x.device) * 2 * np.pi * 1j
        omega2 = torch.fft.fftfreq(self.modes2, d=1/self.modes2).to(x.device) * 2 * np.pi * 1j

        denom = (omega1.view(1, 1, -1, 1) - self.weights_pole1.unsqueeze(-1)) * \
                (omega2.view(1, 1, 1, -1) - self.weights_pole2.unsqueeze(-2))

        H_s = torch.div(self.weights_residue, denom)
        out_freq = torch.einsum("bixy,ioxy->boxy", alpha_modes, H_s)

        return torch.real(torch.fft.ifft2(out_freq, s=target_size, dim=[-2, -1]))

class ChannelMixer(nn.Module):
    def __init__(self, dim, expansion=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim * expansion, 1),
            nn.GELU(),
            nn.Conv2d(dim * expansion, dim, 1)
        )
    def forward(self, x):
        return self.net(x)

class CoDALNOBlock(nn.Module):
    def __init__(self, dim, modes=16, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.InstanceNorm2d(dim)
        self.spatial_mixer = LaplaceSpatialMixer(dim, modes1=modes, modes2=modes)
        self.norm2 = nn.InstanceNorm2d(dim)
        self.channel_mixer = ChannelMixer(dim, expansion=mlp_ratio)
        self.alpha = nn.Parameter(torch.ones(1, dim, 1, 1) * 0.1) # 초기값 조정
        self.beta = nn.Parameter(torch.ones(1, dim, 1, 1) * 0.1)

    def forward(self, x):
        x = x + self.alpha * self.spatial_mixer(self.norm1(x))
        x = x + self.beta * self.channel_mixer(self.norm2(x))
        return x

class CoDALNO_SR(nn.Module):
    def __init__(self, in_channels=3, width=64, blocks=4, modes=16):
        super().__init__()
        self.width = width
        self.lifting = nn.Conv2d(in_channels, width, 1)
        self.scale_embed = nn.Sequential(
            nn.Linear(1, width), nn.GELU(), nn.Linear(width, width)
        )
        self.blocks = nn.ModuleList([CoDALNOBlock(width, modes=modes) for _ in range(blocks)])
        self.upsampler = LaplaceSpatialMixer(width, modes1=modes, modes2=modes)
        self.proj = nn.Conv2d(width, in_channels, 1)

    def forward(self, x, target_size, scale_factor):
        if isinstance(scale_factor, (int, float)):
            scale_factor = torch.full((x.shape[0], 1), scale_factor, dtype=torch.float32, device=x.device)

        feat = self.lifting(x)
        feat = feat + self.scale_embed(scale_factor).view(-1, self.width, 1, 1)

        shortcut = feat
        for block in self.blocks:
            feat = block(feat)
        feat = feat + shortcut

        out = self.proj(self.upsampler(feat, target_size=target_size))
        return out + F.interpolate(x, size=target_size, mode='bicubic', align_corners=False)

In [ ]:
class SpectralLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.L1Loss()
    def forward(self, pred, target):
        p_f = torch.fft.fft2(pred, dim=[-2, -1], norm='ortho')
        t_f = torch.fft.fft2(target, dim=[-2, -1], norm='ortho')
        return self.l1(torch.abs(p_f), torch.abs(t_f))

def get_edge_weight(hr_img, device, alpha=2.0):
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3).to(device)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3).to(device)
    gray = torch.mean(hr_img, dim=1, keepdim=True)
    edge = torch.abs(F.conv2d(gray, sobel_x, padding=1)) + torch.abs(F.conv2d(gray, sobel_y, padding=1))
    return 1.0 + alpha * (edge / (edge.max() + 1e-5))

In [ ]:
import time
import os
import glob
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader

def train(config=None):
    # --- 설정 ---
    BATCH_SIZE = 8
    LR = 1e-4
    EPOCHS = 1000
    SCALE = 4
    VIS_FREQ = 10
    SAVE_LATEST_FREQ = 10
    SAVE_BACKUP_FREQ = 50

    LAMBDA_PIX = 1.0
    LAMBDA_FREQ = 0.1

    print("🚀 Training Setup Started...")

    # 1. Dataset & DataLoader
    dataset = DIV2KDataset(DATA_PATH, phase='train', hr_crop_size=192)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    scales = [2, 3, 4]

    # 2. Model & Optimizer
    model = CoDALNO_SR(width=128, blocks=10, modes=32).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    # 3. Loss Functions
    # vgg_crit는 완전히 제거했습니다.
    freq_crit = SpectralLoss().to(device)

    # ---------------------------------------------------------
    # 검증 이미지 로드 (원래 코드 그대로)
    # ---------------------------------------------------------
    val_img_path = os.path.join(DATA_PATH, 'DIV2K_valid_HR', '0801.png')
    if not os.path.exists(val_img_path):
        val_imgs = sorted(glob.glob(os.path.join(DATA_PATH, 'DIV2K_valid_HR', '*.png')))
        if val_imgs: val_img_path = val_imgs[0]

    if os.path.exists(val_img_path):
        val_hr_pil = Image.open(val_img_path).convert('RGB')
        w, h = val_hr_pil.size
        crop = 512
        if w > crop and h > crop:
            val_hr_pil = val_hr_pil.crop((0, 0, crop, crop))
        w, h = val_hr_pil.size
        w, h = w - (w%2), h - (h%2)

        val_hr_pil = val_hr_pil.resize((w, h), Image.BICUBIC)
        val_lr_pil = val_hr_pil.resize((w//SCALE, h//SCALE), Image.BICUBIC)

        val_hr_tensor = transforms.ToTensor()(val_hr_pil).unsqueeze(0).to(device)
        val_lr_tensor = transforms.ToTensor()(val_lr_pil).unsqueeze(0).to(device)
        val_target_size = (h, w)
    else:
        val_hr_tensor = None
        print("⚠️ 검증용 이미지를 찾을 수 없어 시각화를 건너뜁니다.")

    # ---------------------------------------------------------
    # 학습 이어서 하기 (Resume Logic) 및 스케줄러 셋업
    # ---------------------------------------------------------
    start_epoch = 0
    total_steps = EPOCHS * len(dataloader) # 총 배치 스텝 수 계산

    if not os.path.exists(CKPT_PATH):
        os.makedirs(CKPT_PATH)

    latest_ckpt_path = os.path.join(CKPT_PATH, 'codalno_latest.pth')

    if os.path.exists(latest_ckpt_path):
        print(f"🔄 Found checkpoint at {latest_ckpt_path}. Resuming training...")
        checkpoint = torch.load(latest_ckpt_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        start_epoch = checkpoint['epoch'] + 1
        current_step = start_epoch * len(dataloader)
        print(f"▶ Resuming from Epoch {start_epoch} (Step: {current_step})")
    else:
        current_step = 0
        print("🆕 Starting from scratch.")

    # [핵심 수정] 수만 번 도는 for문 제거하고 last_epoch 파라미터로 즉시 복구!
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=1e-6, last_epoch=current_step - 1
    )

    # ---------------------------------------------------------
    # 🌟 엣지 가중치 함수 (원래 형태 그대로 유지)
    # ---------------------------------------------------------
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3).to(device)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3).to(device)

    def get_edge_weight(hr_img, alpha=3.0):
        gray_hr = torch.mean(hr_img, dim=1, keepdim=True)
        edge_x = F.conv2d(gray_hr, sobel_x, padding=1)
        edge_y = F.conv2d(gray_hr, sobel_y, padding=1)
        edge_map = torch.abs(edge_x) + torch.abs(edge_y)
        weight = 1.0 + alpha * (edge_map / (edge_map.max() + 1e-5))
        return weight

    print("✅ Setup Completed. Start Training...\n")

    # ---------------------------------------------------------
    # 4. Training Loop
    # ---------------------------------------------------------
    model.train()

    for epoch in range(start_epoch, EPOCHS):
        epoch_pix_loss = 0.0
        epoch_freq_loss = 0.0
        epoch_start_time = time.time()

        for i, hr in enumerate(dataloader):
            hr = hr.to(device)
            target_size = (hr.shape[2], hr.shape[3])

            scale = random.choice(scales)
            lr = F.interpolate(hr, scale_factor=1.0/scale, mode='bicubic', align_corners=False)

            if hr.max() > 1.5:
                lr, hr = lr / 255.0, hr / 255.0

            optimizer.zero_grad()
            pred = model(lr, target_size, scale_factor=scale)

            edge_weight = get_edge_weight(hr, alpha=2.0)
            l_pix = torch.mean(edge_weight * torch.abs(pred - hr))
            l_freq = freq_crit(pred, hr)

            loss = (LAMBDA_PIX * l_pix) + (LAMBDA_FREQ * l_freq)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # 스케줄러 매 배치마다 업데이트
            scheduler.step()

            epoch_pix_loss += l_pix.item()
            epoch_freq_loss += l_freq.item()

        avg_pix = epoch_pix_loss / len(dataloader)
        avg_freq = epoch_freq_loss / len(dataloader)
        avg_total = (LAMBDA_PIX * avg_pix) + (LAMBDA_FREQ * avg_freq)
        epoch_time = time.time() - epoch_start_time

        # [수정] 스케줄러에서 정확한 현재 LR을 가져옵니다.
        current_lr = scheduler.get_last_lr()[0]

        print(f"Epoch [{epoch+1}/{EPOCHS}] Time: {epoch_time:.1f}s | LR: {current_lr:.2e} | Total: {avg_total:.5f} (Pix: {avg_pix:.5f}, Freq: {avg_freq:.5f})")

        # 저장 및 시각화 코드 (원래 코드 그대로)
        if (epoch+1) % SAVE_LATEST_FREQ == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict()
            }, latest_ckpt_path)

        if (epoch+1) % SAVE_BACKUP_FREQ == 0:
            backup_path = os.path.join(CKPT_PATH, f'codalno_epoch_{epoch+1}.pth')
            torch.save(model.state_dict(), backup_path)
            print(f"💾 Backup Saved: {backup_path}")

        if (epoch+1) % VIS_FREQ == 0 and val_hr_tensor is not None:
            model.eval()
            with torch.no_grad():
                sr_tensor = model(val_lr_tensor, target_size=val_target_size, scale_factor=SCALE)
                sr_tensor = torch.clamp(sr_tensor, 0.0, 1.0)

            lr_np = val_lr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
            sr_np = sr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
            hr_np = val_hr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()

            plt.figure(figsize=(15, 5))
            plt.subplot(1, 3, 1)
            plt.title(f"LR Input (Epoch {epoch+1})")
            plt.imshow(lr_np)
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.title("CoDALNO SR Output")
            plt.imshow(sr_np)
            plt.axis('off')

            plt.subplot(1, 3, 3)
            plt.title("Ground Truth")
            plt.imshow(hr_np)
            plt.axis('off')

            res_dir = os.path.join(PROJECT_PATH, 'results_codalno')
            os.makedirs(res_dir, exist_ok=True)
            save_img_path = os.path.join(res_dir, f"result_epoch_{epoch+1}.png")

            plt.savefig(save_img_path, bbox_inches='tight')
            plt.show()
            model.train()

# 실행
train(None)